# 03 - Comandos DML em tabelas Delta Lake (MinIO)

Reproduz no bucket **bronze** as 3 operações do trabalho 1, agora com os dados vindos do SQL Server:

| Operação | O que faz |
|---|---|
| **INSERT** | Insere o pedido 106 (Webcam) |
| **UPDATE** | Marca como `PAGO` os pedidos pendentes do cliente 1 |
| **DELETE** | Remove os pedidos `CANCELADO` |

Depois mostra o **histórico** (`DESCRIBE HISTORY`) e o **time travel**.

**Pré-requisito:** notebook `02` executado. Para repetir o experimento, execute o `02` de novo antes.

## 1. SparkSession

In [1]:
import os
from pathlib import Path
from dotenv import load_dotenv

RAIZ = Path.cwd().parent if Path.cwd().name == 'notebook' else Path.cwd()
if not (RAIZ / '.env').exists():
    raise FileNotFoundError('Arquivo .env não encontrado. Rode: cp .env.example .env')
load_dotenv(RAIZ / '.env', override=True)

from pyspark.sql import SparkSession
from delta import *
from delta.tables import DeltaTable

MINIO_ENDPOINT   = os.getenv('MINIO_ENDPOINT')
MINIO_ACCESS_KEY = os.getenv('MINIO_ACCESS_KEY')
MINIO_SECRET_KEY = os.getenv('MINIO_SECRET_KEY')
LANDING_BUCKET   = os.getenv('MINIO_LANDING_BUCKET')
BRONZE_BUCKET    = os.getenv('MINIO_BRONZE_BUCKET')

spark = (
    SparkSession.builder
    .appName('DML_Delta_Lake')
    .master('local[*]')
    .config('spark.jars.packages', 'io.delta:delta-spark_2.12:3.2.1,org.apache.hadoop:hadoop-aws:3.3.4')
    .config('spark.sql.extensions', 'io.delta.sql.DeltaSparkSessionExtension')
    .config('spark.sql.catalog.spark_catalog', 'org.apache.spark.sql.delta.catalog.DeltaCatalog')
    # MinIO / S3A
    .config('spark.hadoop.fs.s3a.endpoint', MINIO_ENDPOINT)
    .config('spark.hadoop.fs.s3a.access.key', MINIO_ACCESS_KEY)
    .config('spark.hadoop.fs.s3a.secret.key', MINIO_SECRET_KEY)
    .config('spark.hadoop.fs.s3a.path.style.access', 'true')
    .config('spark.hadoop.fs.s3a.impl', 'org.apache.hadoop.fs.s3a.S3AFileSystem')
    .config('spark.hadoop.fs.s3a.connection.ssl.enabled', 'false')
    .getOrCreate()
)
print('SparkSession criada! Spark', spark.version)

26/10/12 19:15:06 WARN Utils: Your hostname, DESKTOP-73OKR5D resolves to a loopback address: 127.0.1.1; using 10.255.255.254 instead (on interface lo)
26/10/12 19:15:06 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address


:: loading settings :: url = jar:file:/home/thomasmaggi/pesquisadados_parte2/.venv/lib/python3.11/site-packages/pyspark/jars/ivy-2.5.1.jar!/org/apache/ivy/core/settings/ivysettings.xml


Ivy Default Cache set to: /home/thomasmaggi/.ivy2/cache
The jars for the packages stored in: /home/thomasmaggi/.ivy2/jars
io.delta#delta-spark_2.12 added as a dependency
org.apache.hadoop#hadoop-aws added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-1d1bdde5-215d-48b3-8acb-e59fe93a4bb1;1.0
	confs: [default]
	found io.delta#delta-spark_2.12;3.2.1 in central
	found io.delta#delta-storage;3.2.1 in central
	found org.antlr#antlr4-runtime;4.9.3 in central
	found org.apache.hadoop#hadoop-aws;3.3.4 in central
	found com.amazonaws#aws-java-sdk-bundle;1.12.262 in central
	found org.wildfly.openssl#wildfly-openssl;1.0.7.Final in central
:: resolution report :: resolve 368ms :: artifacts dl 14ms
	:: modules in use:
	com.amazonaws#aws-java-sdk-bundle;1.12.262 from central in [default]
	io.delta#delta-spark_2.12;3.2.1 from central in [default]
	io.delta#delta-storage;3.2.1 from central in [default]
	org.antlr#antlr4-runtime;4.9.3 from central in [default]
	org.a

SparkSession criada! Spark 3.5.3


## 2. Registrar as tabelas Delta como tabelas SQL

Usamos `LOCATION`, apontando para os dados que já estão no MinIO. Isso cria tabelas **não gerenciadas**
(externas): o Spark guarda só os metadados e os dados continuam no bucket `bronze`.

In [2]:
for tabela in ['clientes', 'pedidos']:
    spark.sql(f"""
        CREATE TABLE IF NOT EXISTS {tabela}
        USING delta
        LOCATION 's3a://{BRONZE_BUCKET}/{tabela}'
    """)

spark.sql('SHOW TABLES').show(truncate=False)

26/10/12 19:15:24 WARN MetricsConfig: Cannot locate configuration: tried hadoop-metrics2-s3a-file-system.properties,hadoop-metrics2.properties


+---------+---------+-----------+
|namespace|tableName|isTemporary|
+---------+---------+-----------+
|default  |clientes |false      |
|default  |pedidos  |false      |
+---------+---------+-----------+



## 3. Estado inicial (SELECT)

In [3]:
print('=== CLIENTES ===')
spark.sql('SELECT * FROM clientes ORDER BY id_cliente').show()

print('=== PEDIDOS ===')
spark.sql('SELECT * FROM pedidos ORDER BY id_pedido').show()

# Guarda a versão atual da tabela, para comparar no time travel
VERSAO_INICIAL = DeltaTable.forName(spark, 'pedidos').history(1).collect()[0]['version']
print(f'Versão inicial da tabela pedidos: {VERSAO_INICIAL}')

=== CLIENTES ===


26/10/12 19:15:30 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.
                                                                                

+----------+----------+-------------+---+
|id_cliente|      nome|       cidade| uf|
+----------+----------+-------------+---+
|         1| Ana Souza|     Criciúma| SC|
|         2|Bruno Lima|Florianópolis| SC|
|         3|Carla Dias| Porto Alegre| RS|
|         4|Diego Melo|     Curitiba| PR|
+----------+----------+-------------+---+

=== PEDIDOS ===


+---------+----------+--------+----------+--------------+-----------+---------+
|id_pedido|id_cliente| produto|quantidade|valor_unitario|data_pedido|   status|
+---------+----------+--------+----------+--------------+-----------+---------+
|      101|         1|Notebook|         1|       4500.00| 2026-09-01|     PAGO|
|      102|         2|   Mouse|         2|         80.00| 2026-09-02|     PAGO|
|      103|         3| Monitor|         1|       1200.00| 2026-09-03| PENDENTE|
|      104|         1| Teclado|         1|        250.00| 2026-09-05| PENDENTE|
|      105|         4| Headset|         1|        300.00| 2026-09-06|CANCELADO|
+---------+----------+--------+----------+--------------+-----------+---------+

Versão inicial da tabela pedidos: 0


## 4. INSERT

Insere um novo pedido.

In [4]:
spark.sql("""
    INSERT INTO pedidos VALUES
    (106, 2, 'Webcam', 1, 350.00, DATE'2026-09-10', 'PENDENTE')
""")

spark.sql('SELECT * FROM pedidos ORDER BY id_pedido').show()

+---------+----------+--------+----------+--------------+-----------+---------+
|id_pedido|id_cliente| produto|quantidade|valor_unitario|data_pedido|   status|
+---------+----------+--------+----------+--------------+-----------+---------+
|      101|         1|Notebook|         1|       4500.00| 2026-09-01|     PAGO|
|      102|         2|   Mouse|         2|         80.00| 2026-09-02|     PAGO|
|      103|         3| Monitor|         1|       1200.00| 2026-09-03| PENDENTE|
|      104|         1| Teclado|         1|        250.00| 2026-09-05| PENDENTE|
|      105|         4| Headset|         1|        300.00| 2026-09-06|CANCELADO|
|      106|         2|  Webcam|         1|        350.00| 2026-09-10| PENDENTE|
+---------+----------+--------+----------+--------------+-----------+---------+



## 5. UPDATE

Marca como `PAGO` os pedidos pendentes do cliente 1.

In [5]:
print('ANTES:')
spark.sql('SELECT * FROM pedidos WHERE id_cliente = 1 ORDER BY id_pedido').show()

spark.sql("""
    UPDATE pedidos SET status = 'PAGO'
    WHERE id_cliente = 1 AND status = 'PENDENTE'
""")

print('DEPOIS:')
spark.sql('SELECT * FROM pedidos WHERE id_cliente = 1 ORDER BY id_pedido').show()

ANTES:
+---------+----------+--------+----------+--------------+-----------+--------+
|id_pedido|id_cliente| produto|quantidade|valor_unitario|data_pedido|  status|
+---------+----------+--------+----------+--------------+-----------+--------+
|      101|         1|Notebook|         1|       4500.00| 2026-09-01|    PAGO|
|      104|         1| Teclado|         1|        250.00| 2026-09-05|PENDENTE|
+---------+----------+--------+----------+--------------+-----------+--------+

DEPOIS:


[Stage 46:============================================>           (40 + 8) / 50]

+---------+----------+--------+----------+--------------+-----------+------+
|id_pedido|id_cliente| produto|quantidade|valor_unitario|data_pedido|status|
+---------+----------+--------+----------+--------------+-----------+------+
|      101|         1|Notebook|         1|       4500.00| 2026-09-01|  PAGO|
|      104|         1| Teclado|         1|        250.00| 2026-09-05|  PAGO|
+---------+----------+--------+----------+--------------+-----------+------+



## 6. DELETE

Remove os pedidos cancelados.

In [6]:
print('ANTES:')
spark.sql('SELECT * FROM pedidos ORDER BY id_pedido').show()

spark.sql("DELETE FROM pedidos WHERE status = 'CANCELADO'")

print('DEPOIS:')
spark.sql('SELECT * FROM pedidos ORDER BY id_pedido').show()

ANTES:
+---------+----------+--------+----------+--------------+-----------+---------+
|id_pedido|id_cliente| produto|quantidade|valor_unitario|data_pedido|   status|
+---------+----------+--------+----------+--------------+-----------+---------+
|      101|         1|Notebook|         1|       4500.00| 2026-09-01|     PAGO|
|      102|         2|   Mouse|         2|         80.00| 2026-09-02|     PAGO|
|      103|         3| Monitor|         1|       1200.00| 2026-09-03| PENDENTE|
|      104|         1| Teclado|         1|        250.00| 2026-09-05|     PAGO|
|      105|         4| Headset|         1|        300.00| 2026-09-06|CANCELADO|
|      106|         2|  Webcam|         1|        350.00| 2026-09-10| PENDENTE|
+---------+----------+--------+----------+--------------+-----------+---------+



DEPOIS:


+---------+----------+--------+----------+--------------+-----------+--------+
|id_pedido|id_cliente| produto|quantidade|valor_unitario|data_pedido|  status|
+---------+----------+--------+----------+--------------+-----------+--------+
|      101|         1|Notebook|         1|       4500.00| 2026-09-01|    PAGO|
|      102|         2|   Mouse|         2|         80.00| 2026-09-02|    PAGO|
|      103|         3| Monitor|         1|       1200.00| 2026-09-03|PENDENTE|
|      104|         1| Teclado|         1|        250.00| 2026-09-05|    PAGO|
|      106|         2|  Webcam|         1|        350.00| 2026-09-10|PENDENTE|
+---------+----------+--------+----------+--------------+-----------+--------+



## 7. HISTORY — histórico de versões

Cada operação gera uma nova versão no log de transações do Delta (`_delta_log`).

In [7]:
spark.sql('DESCRIBE HISTORY pedidos').select('version', 'timestamp', 'operation').show(truncate=False)

+-------+-------------------+---------+
|version|timestamp          |operation|
+-------+-------------------+---------+
|3      |2026-10-12 19:15:51|DELETE   |
|2      |2026-10-12 19:15:45|UPDATE   |
|1      |2026-10-12 19:15:40|WRITE    |
|0      |2026-10-12 18:41:07|WRITE    |
+-------+-------------------+---------+



In [8]:
dt = DeltaTable.forName(spark, 'pedidos')
dt.history().select('version', 'operation', 'operationMetrics').show(truncate=False)

+-------+---------+-------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+
|version|operation|operationMetrics                                                                                                                                                                                                                                                                                                               |
+-------+---------+-------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------

## 8. TIME TRAVEL — ler versões anteriores

In [9]:
print(f'=== PEDIDOS - VERSÃO {VERSAO_INICIAL} (antes das operações DML) ===')
df_antes = (spark.read.format('delta')
            .option('versionAsOf', VERSAO_INICIAL)
            .load(f's3a://{BRONZE_BUCKET}/pedidos'))
df_antes.orderBy('id_pedido').show()

=== PEDIDOS - VERSÃO 0 (antes das operações DML) ===


+---------+----------+--------+----------+--------------+-----------+---------+
|id_pedido|id_cliente| produto|quantidade|valor_unitario|data_pedido|   status|
+---------+----------+--------+----------+--------------+-----------+---------+
|      101|         1|Notebook|         1|       4500.00| 2026-09-01|     PAGO|
|      102|         2|   Mouse|         2|         80.00| 2026-09-02|     PAGO|
|      103|         3| Monitor|         1|       1200.00| 2026-09-03| PENDENTE|
|      104|         1| Teclado|         1|        250.00| 2026-09-05| PENDENTE|
|      105|         4| Headset|         1|        300.00| 2026-09-06|CANCELADO|
+---------+----------+--------+----------+--------------+-----------+---------+



In [10]:
df_atual = spark.read.format('delta').load(f's3a://{BRONZE_BUCKET}/pedidos')

print(f'Versão {VERSAO_INICIAL}: {df_antes.count()} registros')
print(f'Atual:      {df_atual.count()} registros')

print('\nRegistros que existem só na versão atual (inseridos ou alterados):')
df_atual.exceptAll(df_antes).orderBy('id_pedido').show()

print('Registros que existiam na versão inicial e foram removidos ou alterados:')
df_antes.exceptAll(df_atual).orderBy('id_pedido').show()

Versão 0: 5 registros
Atual:      5 registros

Registros que existem só na versão atual (inseridos ou alterados):
+---------+----------+-------+----------+--------------+-----------+--------+
|id_pedido|id_cliente|produto|quantidade|valor_unitario|data_pedido|  status|
+---------+----------+-------+----------+--------------+-----------+--------+
|      104|         1|Teclado|         1|        250.00| 2026-09-05|    PAGO|
|      106|         2| Webcam|         1|        350.00| 2026-09-10|PENDENTE|
+---------+----------+-------+----------+--------------+-----------+--------+

Registros que existiam na versão inicial e foram removidos ou alterados:
+---------+----------+-------+----------+--------------+-----------+---------+
|id_pedido|id_cliente|produto|quantidade|valor_unitario|data_pedido|   status|
+---------+----------+-------+----------+--------------+-----------+---------+
|      104|         1|Teclado|         1|        250.00| 2026-09-05| PENDENTE|
|      105|         4|Headse

## 9. Resumo

In [11]:
print('=' * 60)
print('RESUMO DAS OPERAÇÕES DML (tabela pedidos)')
print('=' * 60)
print('INSERT : pedido 106 (Webcam, cliente 2)')
print('UPDATE : pedido 104 do cliente 1 passou de PENDENTE para PAGO')
print('DELETE : pedido 105 (CANCELADO) removido')
print('HISTORY e TIME TRAVEL: cada operação virou uma versão do Delta')
print('=' * 60)

RESUMO DAS OPERAÇÕES DML (tabela pedidos)
INSERT : pedido 106 (Webcam, cliente 2)
UPDATE : pedido 104 do cliente 1 passou de PENDENTE para PAGO
DELETE : pedido 105 (CANCELADO) removido
HISTORY e TIME TRAVEL: cada operação virou uma versão do Delta


In [12]:
spark.stop()
print('SparkSession finalizada.')

SparkSession finalizada.
